# Season-dependent birth rate - Herpeirsus Example

This notebook contains code for simulating a Herpeirsus virus-type epidemic in a toy rodent population. This rodent epidemic follows the dynamics of the SIR algorithm with contant birth term rate, whose value depends on the time of the year, as described by _[1]_. The produced data are meant to replicate the dynamics of wildlife rodent population, on which realistic field data collection studies can be performed. These studies collect metaviromic data from randomly selected rodents.

Therefore, in this notebook we also include code for drawing random samples of rodents at predifined sampling times, which satisfy the following:
 - same total number of rodents sampled at each time point;
 - the sampled individuals can be either susceptible (S), infected (I) or recovered (R), with no predefined quantities of each;
 - all individuals sampled are born and alive at the time of sampling.

For each of the sampled individuals, we use the SIR model's embedded `viral_read_model` to produce viral read count data, similar to what data is produced from the field studies (byproduct in our analyses, ground truth in real studies). Replicating the analyses performed in _James Hay et al. (2021)[2]_, we determine whether the skweness and mean of the sampled viral read data at each sampled point accurately reflect the rise and fall of the toy epidemic.

**************
### References
[1] Diana Erazo et al., _Who acquires infection from whom? Estimating herpesvirus transmission rates between wild rodent host groups_. Epidemics35(2021). DOI:10.1016/j.epidem.2021.100451

[2] James A. Hay et al., _Estimating epidemiologic dynamics from cross-sectional viral load distributions_. Science373,**eabh0635(2021)**. DOI:10.1126/science.abh0635

In [1]:
# Load necessary libraries
import os
import numpy as np
import pandas as pd
from scipy.stats import multinomial, skew
import math
import metavirommodel as mm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Choose array of colours for graphs and compartments names
colours = ['blue', 'red', 'green', 'purple', 'orange', 'black', 'gray', 'pink']
compartments = ['S', 'I', 'R']

## Gillespie stochastic SIR algorithm with contant _precipitation-dependent_ birth term rate 

#### Define rodent population

In [2]:
# Set initial reproduction number
R_0 = 5

# Set initial population state S - I - R
N_init = 400
# S_init = int(N_init / R_0)
S_init = 380
I_init = N_init - S_init
R_init = 0
initial_population = [S_init, I_init, R_init]

# Set birth rate
theta = mm.BirthRateSeason([0.539, 0.0148])

# Set death rates
mu = 0.00037
nu = 0.0

# Set transition rates
infect_period = 14
# beta = R_0 / infect_period
# gamma = 1 / infect_period

beta = 1.45 * 10**-4 * N_init
gamma = 0.0457

# Coalesce into paramater vector
parameters = initial_population
parameters.extend([theta, mu, nu, beta, gamma])

# Instantiate algorithm
algorithm = mm.Metaviromodel()

# Select start and end times
start_time = 1
end_time = 360

times = list(range(start_time, end_time+1))

# Select number of experiments
num_experiments = 1

output_algorithm = []

S_history_algorithm = []
I_history_algorithm = []
R_history_algorithm = []

I_times_history_algorithm = []
R_times_history_algorithm = []

for _ in range(num_experiments):
    output, S_history, I_history, R_history, I_times_history, R_times_history = algorithm.simulate_fixed_times(parameters, start_time, end_time)
    output_algorithm.append(output)

    S_history_algorithm.append(S_history)
    I_history_algorithm.append(I_history)
    R_history_algorithm.append(R_history)

    I_times_history_algorithm.append(I_times_history)
    R_times_history_algorithm.append(R_times_history)

output_algorithm = np.asarray(output_algorithm)

#### Plot season-dependent growth rate

In [3]:
# Birth rate
fig = go.Figure()

fig.add_trace(
        go.Scatter(
            y=[theta(t) for t in np.arange(0, 365)],
            x=np.arange(0, 365),
            mode='lines',
            name='Birth rate'
        ))

fig.show()

### Plot output of Gillespie for the different compartments
#### One iteration

In [4]:
# Trace names - represent the type of individuals for the simulation
trace_name = ['{}'.format(s) for s in compartments]

# Names of panels
panels = ['{} only'.format(s) for s in compartments] + ['Total Population']

fig = go.Figure()
fig = make_subplots(rows=int(np.ceil(len(panels)/2)), cols=2, subplot_titles=tuple('{}'.format(p) for p in panels))

# Add traces to the separate counts panels
for s, spec in enumerate(compartments):
    fig.add_trace(
        go.Scatter(
            y=np.mean(output_algorithm[:, :, s], axis=0).tolist(),
            x=times,
            mode='lines',
            name=trace_name[s],
            line_color=colours[s]
        ),
        row= int(np.floor(s / 2)) + 1,
        col= s % 2 + 1
    )

fig.add_trace(
    go.Scatter(
        y=np.mean(np.sum(output_algorithm, axis=2), axis=0).tolist(),
        x=times,
        mode='lines',
        name='Total Population',
        line_color='black'
    ),
    row= 2,
    col= 2
)

# Add axis labels
fig.update_layout(
    title='Counts of compartments over time:<br>IC = {}, θ = {}, μ = {}, v = {}, β = {:.2f}, γ = {:.2f}'.format(parameters[0:3], parameters[3], parameters[4], parameters[5], parameters[6], parameters[7]),
    width=1100, 
    height=600,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis2=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis2=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis3=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis3=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis4=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis4=dict(
        linecolor='black',
        title = 'Individuals')
    )

fig.write_image('images/SIR-gillespie-Herpeirsus.pdf')
fig.show()

#### All iterations

In [5]:
# Trace names - represent the type of individuals for the simulation
trace_name = ['{}'.format(s) for s in compartments]

# Names of panels
panels = ['{} only'.format(s) for s in compartments] + ['Total Population']

fig = go.Figure()
fig = make_subplots(rows=int(np.ceil(len(panels)/2)), cols=2, subplot_titles=tuple('{}'.format(p) for p in panels))

# Add traces to the separate counts panels
for s, spec in enumerate(compartments):
    for _ in range(num_experiments):
        fig.add_trace(
            go.Scatter(
                y=output_algorithm[_, :, s].tolist(),
                x=times,
                mode='lines',
                name=trace_name[s],
                line_color=colours[s],
                showlegend=False,
            ),
            row= int(np.floor(s / 2)) + 1,
            col= s % 2 + 1
        )

fig.add_trace(
    go.Scatter(
        y=np.mean(np.sum(output_algorithm, axis=2), axis=0).tolist(),
        x=times,
        mode='lines',
        name='Total Population',
        line_color='black'
    ),
    row= 2,
    col= 2
)

# Add axis labels
fig.update_layout(
    title='Counts of compartments over time:<br>IC = {}, θ = {}, μ = {}, v = {}, β = {:.2f}, γ = {:.2f}'.format(parameters[0:3], parameters[3], parameters[4], parameters[5], parameters[6], parameters[7]),
    width=1100, 
    height=600,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis2=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis2=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis3=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis3=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis4=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis4=dict(
        linecolor='black',
        title = 'Individuals')
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/SIR-gillespie-separate-Herpeirsus.pdf')
fig.show()

## Produce Viral read counts values

In [6]:
# Set parameter for the viral read counts model
t_eclipse = 3  # (0 days) Time from infection to initial viral growth
t_peak = 10  # (5 days ) Time from initial viral growth to peak viral load
t_switch = 15  # (9.38 days) Time from peak viral load to secondary waning phase
t_mod = 30  # (14 days) time from secondary waning phase until gumbel distribution reaches its min scale parameter
t_LOD = math.inf  # ( inf days ) Time from infection until modal read counts value is equal to the limit of detection

sigma_obs = 5  # Initial scale parameter for the Gumbel distribution until a=teclipse+tpeak+tswitch
s_mod = 0.4  # 0.4 multiplicative factor applied to scale paramter for the Gumble distrbution - starting at t_eclipse + t_peak + t_switch + t_scle
v_zero = 6  # read counts value at time of infection
v_peak = 388  # (20) Modal read counts value at peak viral load
v_switch = 140  # (33) Modal read counts value at a = teclipse + tpeak + tswitch
v_LOD = 18  # Limit of detection of read counts value

parameters_vl = [
    t_eclipse, t_peak, t_switch, t_mod, t_LOD,
    v_zero, v_peak, v_switch, v_LOD,
    s_mod, sigma_obs]

# Set read counts value for the suceptible and recovered individuals
VR_susc = 6

### Plot Viral read Model

In [7]:
time_from_infec = np.arange(1, 50)
vr_val = []

for ti in time_from_infec:
    ti_vr_val = []
    for _ in range(10000):
        ti_vr_val.append(algorithm.viral_read_model(parameters_vl, ti))
    vr_val.append(ti_vr_val)

vr_val = np.asarray(vr_val)

In [8]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=time_from_infec,
        x=np.mean(vr_val, axis=1),
        mode='lines',
        name='Mean Viral read',
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        y=time_from_infec.tolist() + time_from_infec.tolist()[::-1],
        x=np.quantile(vr_val, 0.975, axis=1).tolist() + np.quantile(vr_val, 0.025, axis=1).tolist()[::-1],
        mode='lines',
        fill='toself',
        fillcolor='blue',
        line_color='blue',
        opacity=0.3,
        showlegend=False,
    )
)

# Add axis labels
fig.update_layout(
    width=500, 
    height=500,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Mean Viral read',
        autorange='reversed'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Time since infection'),
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Viral_read_model_Herpeirsus.pdf')
fig.show()

### Compute the history of recovered individuals that fully clear the virus

#### 0 = 'not cleared'; 1 = 'cleared'

In [9]:
# Daily probability of recovered fully clearing the virus
p_addl = 0.2

R_history_clear_algorithm = []

# Go through each run experiment
for _ in range(num_experiments):
    R_history_clear = []

    # Go through each recorded day
    for t, time in enumerate(times):
        current_clear_status = []

        # If there are any recovered individual
        if len(R_times_history_algorithm[_][t]) > 0:
            # Go through each of them and
            for ind, ind_ID in enumerate(R_history_algorithm[_][t]):
                clear_status = 0

                # If they have previously cleared the virus they signal that
                if ind_ID in R_history_algorithm[_][t-1] and R_history_clear[-1][R_history_algorithm[_][t-1].index(ind_ID)] == 1:
                    clear_status = 1
                # if not, they could do it today, if their time since infection exceeds teclipse + tpeak + tswitch
                elif time > R_times_history_algorithm[_][t][ind] + t_eclipse + t_peak + t_switch:
                    clear_status = 1 - np.random.binomial(1, p = (1-p_addl)**(
                        time - R_times_history_algorithm[_][t][ind] - t_eclipse - t_peak - t_switch))

                current_clear_status.append(clear_status)

        R_history_clear.append(current_clear_status)                

    R_history_clear_algorithm.append(R_history_clear)

### Sample individuals at specific points in time

In [10]:
# Sample indviduals on days 10 to 315
sample_points = np.array([10, 20, 30, 65, 115, 165, 215, 265, 315])

# Select sample size
sample_size = 100

vr_values = []
vr_infec = []

vr_susc_ids = []
vr_infec_ids = []
vr_recov_ids = []

vr_time_of_recov_infec = []
vr_time_of_infec = []
vr_time_since_infec = []

for _ in range(num_experiments):
    experiment_vr_values = []
    experiment_infec = []

    experiment_susc_ids = []
    experiment_infec_ids = []
    experiment_recov_ids = []

    experiment_time_of_recov_infec = []
    experiment_time_of_infec = []
    experiment_time_since_infec = []
    for time in sample_points:
        # Identify the current infections at the specified timepoint
        current_susceptibles = S_history_algorithm[_][time-1]
        current_infections = I_history_algorithm[_][time-1]
        current_recovered = R_history_algorithm[_][time-1]
        current_infection_times = I_times_history_algorithm[_][time-1]
        current_recov_infection_times = R_times_history_algorithm[_][time-1]
        current_recov_clear_virus_status = R_history_clear_algorithm[_][time-1]

        # Sample without replacement the sample_size individuals and
        # determine their time since infection to produce Ct values
        number_selected_susc, number_selected_infec, number_selected_rec = \
            multinomial.rvs(
                n=sample_size,
                p=output_algorithm[_, time-1, :]/np.sum(output_algorithm[_, time-1, :])) # determine how many of those sampled are S, I and R

        # First add the Ct values for the sampled susceptibele and recovered individuals
        sampled_vr_values = [VR_susc] * number_selected_susc

        selected_individuals_susc_ids = np.random.choice(
                current_susceptibles,
                size=number_selected_susc,
                replace=False).tolist() # determine the ids of those sampled Ss
        
        if len(current_recov_infection_times) > 0:
            # If we have at least one selected recovered
            selected_individuals_indices = np.random.choice(
                range(len(current_recov_infection_times)),
                size=number_selected_rec,
                replace=False).tolist() # determine the indices of those sampled Rs
        
            selected_individuals_rec_ids = [current_recovered[_] for _ in selected_individuals_indices]

            # Determine the time of infection of those sampled Rs
            selected_individuals_recov_infec_times = [current_recov_infection_times[_] for _ in selected_individuals_indices]

            sample_time_since_infec = time - selected_individuals_recov_infec_times # determine how long since infection for selected Rs

            # Determine the clearence of infection of those sampled Rs
            selected_individuals_clear_virus_status = [current_recov_clear_virus_status[_] for _ in selected_individuals_indices]

            # Run viral read model to determine individual viral read counts for each sample
            for i, ti in enumerate(sample_time_since_infec):
                sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, ti) *(1 - selected_individuals_clear_virus_status[i]))

        elif number_selected_rec > 0:
            # If initial step when no history of infection is provided
            for i in range(number_selected_rec):
                sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, time))
            
            sample_time_since_infec = np.zeros(number_selected_rec)
            selected_individuals_rec_ids = [] 
        else:
            sample_time_since_infec = []
            selected_individuals_rec_ids = []

        if len(current_infection_times) > 0:
            # If we have at least one selected infection
            selected_individuals_indices = np.random.choice(
                range(len(current_infection_times)),
                size=number_selected_infec,
                replace=False).tolist() # determine the indices of those sampled Is
            
            # Determine the ids of those sampled Is
            selected_individuals_infec_ids = [current_infections[_] for _ in selected_individuals_indices]

            # Determine the time of infection of those sampled Is
            selected_individuals_infec_times = [current_infection_times[_] for _ in selected_individuals_indices]
        
            sample_time_since_infec = time - selected_individuals_infec_times # determine how long since infection for selected Is

            # Run Ct model to determine individual Viral read counts for each sample
            for ti in sample_time_since_infec:
                sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, ti))
        
        elif number_selected_infec > 0:
            # If initial step when no history of infection is provided
            for i in range(number_selected_infec):
                sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, time))
            
            selected_individuals_infec_times = np.zeros(number_selected_infec)
            sample_time_since_infec = np.zeros(number_selected_infec)
            selected_individuals_infec_ids = [] 
        else:
            selected_individuals_infec_times = []
            sample_time_since_infec = []
            selected_individuals_infec_ids = [] 

        experiment_vr_values.append(sampled_vr_values)
        experiment_infec.append(number_selected_infec)
        
        experiment_susc_ids.append(selected_individuals_susc_ids)
        experiment_infec_ids.append(selected_individuals_infec_ids)
        experiment_recov_ids.append(selected_individuals_rec_ids)

        experiment_time_of_recov_infec.append(selected_individuals_recov_infec_times)
        experiment_time_of_infec.append(selected_individuals_infec_times)
        experiment_time_since_infec.append(sample_time_since_infec)
    
    vr_values.append(experiment_vr_values)
    vr_infec.append(experiment_infec)

    vr_susc_ids.append(experiment_susc_ids)
    vr_infec_ids.append(experiment_infec_ids)
    vr_recov_ids.append(experiment_recov_ids)

    vr_time_of_recov_infec.append(experiment_time_of_recov_infec)
    vr_time_of_infec.append(experiment_time_of_infec)
    vr_time_since_infec.append(experiment_time_since_infec)

vr_values = np.asarray(vr_values)
vr_infec = np.asarray(vr_infec)

### Collect all information about infection time in pd.Dataframe

In [11]:
vr_time_of_infec_data = []

for _ in range(num_experiments):
    experiment_vr_time_of_infec_data = pd.DataFrame(columns=['ID', 'Value'])
    for t, time in enumerate(sample_points):
        experiment_vr_time_of_infec_data = pd.concat(
            [
                experiment_vr_time_of_infec_data,
                pd.DataFrame({
                    'ID': vr_susc_ids[_][t] + vr_recov_ids[_][t] + vr_infec_ids[_][t],
                    'Value': [400] * len(vr_susc_ids[_][t]) + vr_time_of_recov_infec[_][t] + vr_time_of_infec[_][t]
                })
            ])
        
    vr_time_of_infec_data.append(experiment_vr_time_of_infec_data)

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_24998/2006244478.py:6: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



### Collect all information about viral read counts in pd.Dataframe

In [12]:
viral_read_data = []

for _ in range(num_experiments):
    experiment_viral_read_data = pd.DataFrame(columns=['ID', 'TimeOfSample', 'Value'])
    for t, time in enumerate(sample_points):
        experiment_viral_read_data = pd.concat(
            [
                experiment_viral_read_data,
                pd.DataFrame({
                    'ID': vr_susc_ids[_][t] + vr_recov_ids[_][t] + vr_infec_ids[_][t],
                    'TimeOfSample': [time] * sample_size,
                    'Value': vr_values[_, t, :].tolist()
                })
            ])
        
    viral_read_data.append(experiment_viral_read_data)

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_24998/3329008490.py:6: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



### Plot Viral read counts per sample time
Coloured by skewness of sample. This replicates analyses from James Hay et al. (2021)

In [13]:
fig = go.Figure()
fig = make_subplots(rows=2, cols=1, subplot_titles=('Average Viral read value', 'Average Prevalence of Infections'))

skews = skew(vr_values[0, :, :], axis=1)

for _ in range(sample_size):
    fig.add_trace(
        go.Scatter(
            y=vr_values[0, :, _].tolist(),
            x=sample_points,
            mode='markers',
            name='Average Viral read value',
            marker_line=dict(width=1.25, color='black'),
            marker_color=skews,
            marker_colorscale='BlueRed',
            marker_colorbar_title=dict(
                text='Skew',
                side='top'),
            marker_size=12,
            marker_opacity=0.6,
            marker_showscale=True,
            showlegend=False,
        ),
        row=1,
        col=1
    )

fig.add_trace(
    go.Scatter(
        y=np.mean(vr_values[0, :, :], axis=1).tolist(),
        x=sample_points,
        mode='markers',
        name='Median Ct value',
        marker_color='black',
        marker_line=dict(width=4, color='black'),
        marker_symbol='line-ew',
        marker_size=16,
        showlegend=False,
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        y=(vr_infec[0, :]/sample_size).tolist(),
        x=sample_points,
        mode='lines+markers',
        connectgaps=True,
        name='Average Prevalence of Infections',
        marker_line=dict(width=1.25, color='black'),
        marker_color=skews,
        marker_colorscale='BlueRed',
        marker_colorbar_title=dict(
            text='Skew',
            side='top'),
        marker_opacity=0.6,
        marker_size=12,
        marker_showscale=True,
        line_color='black',
        showlegend=False,
    ),
    row=2,
    col=1
)

# Add axis labels
fig.update_layout(
    width=1100, 
    height=800,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        tickvals=sample_points.tolist(),
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Viral read value (e+03)'),
    xaxis2=dict(
        linecolor='black',
        tickvals=sample_points.tolist(),
        title = 'Time (days)'
        ),
    yaxis2=dict(
        linecolor='black',
        tickformat = '.0%',
        title = 'Prevalence of Infections<br>in sample')
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Viral_read_SIR_Skewness-Herpeirsus.pdf')
fig.show()

### Plot skenewss and median

In [14]:
fig = go.Figure()
fig = make_subplots(rows=2, cols=1, subplot_titles=('Median time since infection', 'Skewness Viral reads'))

median_ti = []

for ti in vr_time_since_infec[0]:
    median_ti.append(np.median(ti))


fig.add_trace(
    go.Scatter(
        y=median_ti,
        x=sample_points,
        mode='lines+markers',
        connectgaps=True,
        name='Median times',
        marker_line=dict(width=1.25, color='black'),
        marker_opacity=0.6,
        marker_size=12,
        line_color='black',
        showlegend=False,
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        y=skews,
        x=sample_points,
        mode='lines+markers',
        connectgaps=True,
        name='Skewness viral read values',
        marker_line=dict(width=1.25, color='black'),
        marker_opacity=0.6,
        marker_size=12,
        line_color='black',
        showlegend=False,
    ),
    row=2,
    col=1
)

# Add axis labels
fig.update_layout(
    width=1100, 
    height=800,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        tickvals=sample_points.tolist(),
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Median Time since infection'),
    xaxis2=dict(
        linecolor='black',
        tickvals=sample_points.tolist(),
        title = 'Time (days)'
        ),
    yaxis2=dict(
        linecolor='black',
        title = 'Skewness viral read values')
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Median_and_Skewness_Viral_Read-Herpeirsus.pdf')
fig.show()

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning:

Mean of empty slice.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/numpy/_core/_methods.py:145: RuntimeWarning:

invalid value encountered in scalar divide



In [15]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=np.mean(vr_values[0, :, :], axis=1).tolist(),
        x=skews,
        mode='markers',
        name='Mean Viral Read',
        marker_color=sample_points,
        marker_colorscale='BlueRed',
        marker_colorbar_title=dict(
            text='Sample time',
            side='top'),
        marker_showscale=True,
        showlegend=False,
    )
)

# Add axis labels
fig.update_layout(
    width=500, 
    height=500,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Skewness Viral reads'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Mean Viral Read')
    )

fig.write_image('images/Mean_vs_Skewness_Viral_Read-Herpeirsus.pdf')
fig.show()